# Fraud operating reality and past-only feature construction

**Research question:** How can transaction, identity, and behavioural signals be built without future leakage?

This notebook is the review interface. Reusable transformations remain in `src/`, `scripts/`, and `sql/`; the notebook reads their committed evidence rather than creating a second implementation. Raw source data is intentionally not embedded.

## Hypotheses declared before interpretation

- Fraud operations should be evaluated at a daily capacity, not only by global AUC.
- Same-timestamp transactions must share identical pre-batch history.
- Identity coverage and proxy-entity quality are governance constraints.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(f'Repository: {ROOT.name}')


Repository: fraud-ops-intelligence-lab


## Feature provenance

Every row states its raw source, transformation, decision-time cutoff, fit scope, missing-value treatment, and known leakage or proxy risk.

In [2]:
features = pd.read_csv(ROOT / 'reports' / 'feature_provenance.csv')
display(features)
display(features.groupby(['feature_family', 'model_role']).size().rename('features').reset_index())


,feature_name,feature_family,raw_source,grain,transformation,observation_cutoff,available_at_decision,fit_partition,missing_treatment,model_role,exclusion_reason,leakage_risk,proxy_or_fairness_risk,validation
0,TransactionAmt,transaction core,train_transaction.TransactionAmt,transaction,Raw amount,current transaction,yes,none,median in baseline/challenger,model,NaN,low,amount may proxy customer/channel mix,temporal OOT metrics
1,ProductCD/card/address/email/device core,transaction core,train_transaction,transaction,Selected numeric/categorical source fields,current transaction,yes,development,median/most-frequent then ordinal encoding,model,NaN,medium: anonymous semantics,possible geography/device proxies,feature-family ablation
2,has_identity,identity,train_identity join,transaction,1 when identity row exists,current transaction,yes,none,zero,research challenger,NaN,medium: availability process,channel/coverage proxy,identity ablation and uncertainty
3,DeviceType/id_31,identity,train_identity,transaction,Categorical identity values,current transaction,yes,development,most-frequent/unknown encoding,research challenger,NaN,medium,device/browser proxy,parsimony gate; not selected
4,entity_prior_tx_count,past behaviour,proxy entity transaction history,transaction,Count transactions in strictly earlier timesta...,strictly before TransactionDT,yes,none,zero,research challenger,NaN,low after batch-safe grouping,depends on unverified entity proxy,same-timestamp unit test
5,entity_prior_amount_mean,past behaviour,proxy entity transaction history,transaction,Mean amount over strictly earlier timestamp gr...,strictly before TransactionDT,yes,none,missing for new entity,research challenger,NaN,low,depends on unverified entity proxy,same-timestamp unit test
6,entity_seconds_since_previous,past behaviour,proxy entity transaction history,transaction,Elapsed seconds since previous timestamp group,strictly before TransactionDT,yes,none,missing for new entity,research challenger,NaN,low,depends on entity resolution,same-timestamp unit test
7,amount_to_entity_prior_mean,past behaviour,current amount and past-only mean,transaction,TransactionAmt / entity_prior_amount_mean,past history plus current amount,yes,none,missing when no prior mean,research challenger,NaN,low,depends on entity resolution,ablation
8,entity_is_new,past behaviour,entity_prior_tx_count,transaction,1 when no strictly prior transaction exists,strictly before TransactionDT,yes,none,zero,research challenger,NaN,low,newness depends on observation window,ablation
9,entity_key,entity proxy,card/address/email components,transaction,Normalized string concatenation,current transaction,proxy only,none,explicit missing token,queue deduplication,NaN,medium: merges/splits are unverified,identity resolution risk,proxy sensitivity required before production


,feature_family,model_role,features
0,entity proxy,queue deduplication,1
1,identity,research challenger,2
2,past behaviour,research challenger,5
3,transaction core,model,2


## Decision-time audit

A feature is eligible only when its information exists at the stated observation cutoff. Model-fitted bins, encoders, imputers, and thresholds use development data only. Outcomes and evaluation groups never enter the predictor matrix.

In [3]:
audit = features[['feature_name', 'observation_cutoff', 'available_at_decision', 'fit_partition', 'leakage_risk', 'validation']]
display(audit)


,feature_name,observation_cutoff,available_at_decision,fit_partition,leakage_risk,validation
0,TransactionAmt,current transaction,yes,none,low,temporal OOT metrics
1,ProductCD/card/address/email/device core,current transaction,yes,development,medium: anonymous semantics,feature-family ablation
2,has_identity,current transaction,yes,none,medium: availability process,identity ablation and uncertainty
3,DeviceType/id_31,current transaction,yes,development,medium,parsimony gate; not selected
4,entity_prior_tx_count,strictly before TransactionDT,yes,none,low after batch-safe grouping,same-timestamp unit test
5,entity_prior_amount_mean,strictly before TransactionDT,yes,none,low,same-timestamp unit test
6,entity_seconds_since_previous,strictly before TransactionDT,yes,none,low,same-timestamp unit test
7,amount_to_entity_prior_mean,past history plus current amount,yes,none,low,ablation
8,entity_is_new,strictly before TransactionDT,yes,none,low,ablation
9,entity_key,current transaction,proxy only,none,medium: merges/splits are unverified,proxy sensitivity required before production


## Committed analytical evidence

In [4]:
evidence_files = ["reports/data_profile_summary.json", "reports/feature_family_ablation.csv"]
for relative in evidence_files:
    path = ROOT / relative
    display(Markdown(f'### `{relative}`'))
    if not path.exists():
        display(Markdown('*Not committed; reproduce locally using the README pipeline.*'))
    elif path.suffix == '.csv':
        display(pd.read_csv(path).head(20))
    elif path.suffix == '.json':
        payload = json.loads(path.read_text())
        display(payload)


### `reports/data_profile_summary.json`

{'elapsed_days': 182,
 'columns_transaction': 394,
 'features_identity': 40,
 'features_over_90pct_missing': 11,
 'predictors_transaction': 390,
 'fraud_cases': 20663,
 'fraud_exposure': 3083844.8600000003,
 'fraud_exposure_share': 0.03867426030720168,
 'fraud_rate': 0.03499000914417313,
 'fully_missing_features': 0,
 'identity_coverage': 0.2442391709283029,
 'repeat_entity_transaction_share': 0.9210739323331188,
 'top_1pct_fraud_entity_case_share': 0.24705996225136717,
 'total_exposure': 79738948.735,
 'transactions': 590540,
 'unique_proxy_entities': 94873}

### `reports/feature_family_ablation.csv`

,model,mean_precision_at_k,mean_recall_at_k,mean_exposure_captured_at_k,total_legitimate_exposure_interrupted,average_precision,roc_auc,brier_score
0,gbm_plus_behaviour,0.310435,0.365361,0.376903,272470.439,0.401026,0.879540,0.025442
1,gbm_plus_identity,0.313913,0.370007,0.373154,268419.142,0.404105,0.883387,0.025257
2,gbm_transaction,0.310870,0.364957,0.369001,263138.803,0.398293,0.881363,0.025426


## Decision

**Past-only history is valid for research, but proxy-entity linkage must be independently validated before production deduplication.**

### Claim boundary

The notebook distinguishes observed evidence from model output, simulated policy, and causal impact. A strong offline metric is not production exposure.